In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Joint fit: PbSO₄, neutron and X-ray

Fit PbSO₄ to two patterns at once, the Rietveld round robin's neutron pattern from D1A at ILL
and its laboratory X-ray pattern: one structure, two experiments.

The project is the catalogue case `pd-neut-xray-cwl_pbso4-d1a-lab`, the one the command line and
the app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Joint</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">xray</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [ ]:
import easydiffraction as edi

## Load the project

In [ ]:
project_path = edi.download_data('pd-neut-xray-cwl_pbso4-d1a-lab', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['pbso4']
d1a = project.experiments['d1a']
lab = project.experiments['lab']

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [ ]:
for parameter in project.free_parameters:
    parameter.free = False

In [ ]:
project.display.pattern('d1a')

## Perform the analysis

Both patterns fit together. diffraction-lib's tutorial of this case frees everything at once,
and also the D1A wavelength, which this case keeps fixed.

### Scales, 2θ zeros and cell

In [ ]:
d1a.linked_structures['pbso4'].scale.free = True
lab.linked_structures['pbso4'].scale.free = True
d1a.instrument.calib_twotheta_offset.free = True
lab.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Backgrounds

D1A's points and the laboratory pattern's Chebyshev terms.

In [ ]:
for point in d1a.background:
    point.intensity.free = True
for term in lab.background_terms:
    term.coef.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Peak widths

In [ ]:
d1a.peak.broad_gauss_u.free = True
d1a.peak.broad_gauss_v.free = True
d1a.peak.broad_gauss_w.free = True
d1a.peak.mixing_eta_0.free = True
lab.peak.broad_gauss_u.free = True
lab.peak.broad_gauss_v.free = True
lab.peak.broad_gauss_w.free = True
lab.peak.mixing_eta_0.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Peak asymmetry, D1A

The D1A pattern's four Bérar-Baldinozzi terms.

In [ ]:
d1a.peak.asym_beba_a0.free = True
d1a.peak.asym_beba_b0.free = True
d1a.peak.asym_beba_a1.free = True
d1a.peak.asym_beba_b1.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Peak asymmetry, laboratory

The laboratory pattern's first-order terms.

In [ ]:
lab.peak.asym_beba_a0.free = True
lab.peak.asym_beba_b0.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Atom positions

The neutron and X-ray patterns together see the oxygen as well as the lead, so all free
coordinates fit at once.

In [ ]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

### Displacements

In [ ]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('d1a')

In [ ]:
project.display.pattern('lab')

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.structure('pbso4')

## Save the project

In [ ]:
project.save_as('pd-neut-xray-cwl_pbso4-d1a-lab')
project.report.save_html('pd-neut-xray-cwl_pbso4-d1a-lab.html')